LangChain

In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

ImportError: Could not import langchain: Please install langchain extension.

In [3]:
model_id = 'ibm/granite-3-2-8b-instruct' 

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5, # this randomness or creativity of the model's responses
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
}

project_id = "skills-network"

model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)

'`api_key` for IAM token is not provided in credentials for the client'


WMLClientError: '`api_key` for IAM token is not provided in credentials for the client'

In [4]:
msg = model.generate("In today's sales meeting, we ")
print(msg['results'][0]['generated_text'])

NameError: name 'model' is not defined

In [ ]:
granite_llm = WatsonxLLM(model = model)

In [ ]:
print(granite_llm.invoke("Who is man's best friend?"))

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [2]:
msg = granite_llm.invoke(
    [
        SystemMessage(content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"),
        HumanMessage(content="I enjoy mystery novels, what should I read?")
    ]
)
print(msg)

NameError: name 'granite_llm' is not defined

In [5]:
msg = granite_llm.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)
print(msg)

NameError: name 'granite_llm' is not defined

In [6]:
msg = granite_llm.invoke(
    [
        HumanMessage(content="What month follows June?")
    ]
)
print(msg)

NameError: name 'granite_llm' is not defined

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input_ = {"adjective": "funny", "topic": "cats"}  # create a dictionary to store the corresponding input to placeholders in prompt template

In [8]:
prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

input_ = {"topic": "cats"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about cats', additional_kwargs={}, response_metadata={})])

In [10]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")
])

input_ = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the day after Tuesday?', additional_kwargs={}, response_metadata={})])

In [11]:
chain = prompt | granite_llm
response = chain.invoke(input = input_)
print(response)

NameError: name 'granite_llm' is not defined

In [12]:
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Examples of a pretend task of creating antonyms.
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input": "windy", "output": "calm"},
]

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25,  # The maximum length that the formatted examples should be.
)
dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Give the antonym of every input",
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

In [13]:
print(dynamic_prompt.format(adjective="big"))

Give the antonym of every input

Input: happy
Output: sad

Input: tall
Output: short

Input: energetic
Output: lethargic

Input: sunny
Output: gloomy

Input: windy
Output: calm

Input: big
Output:


In [14]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(adjective=long_string))

Give the antonym of every input

Input: happy
Output: sad

Input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
Output:


In [15]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

ModuleNotFoundError: No module named 'langchain_core.pydantic_v1'

In [16]:
# And a query intented to prompt a language model to populate the data structure.
joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | granite_llm | output_parser

chain.invoke({"query": joke_query})

NameError: name 'Joke' is not defined

In [17]:
from langchain.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | granite_llm | output_parser

ModuleNotFoundError: No module named 'langchain.output_parsers'

In [18]:
chain.invoke({"subject": "ice cream flavors"})

NameError: name 'chain' is not defined

In [20]:
from langchain_core.documents import Document

Document(page_content="""Python is an interpreted high-level general-purpose programming language. 
                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
         metadata={
             'my_document_id' : 234234,
             'my_document_source' : "About Python",
             'my_document_create_time' : 1680013019
         })

Document(metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}, page_content="Python is an interpreted high-level general-purpose programming language. \n                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

In [21]:
Document(page_content="""Python is an interpreted high-level general-purpose programming language. 
                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.""")

Document(metadata={}, page_content="Python is an interpreted high-level general-purpose programming language. \n                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

In [22]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")


In [23]:
document = loader.load()
document[2]  # take a look at the page 2

NameError: name 'loader' is not defined

In [24]:
print(document[1].page_content[:1000])  # print the page 1's first 1000 tokens

NameError: name 'document' is not defined

In [25]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [26]:
web_data = loader.load()

In [27]:
print(web_data[0].page_content[:1000])

LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageLangChain + LangGraphSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewLangChainLangGraphDeep AgentsIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewBuilt-in middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain is an open source framework with a pre-built agent architecture and integrations for any model or tool — so you can build agents that adapt as fast as the ecosystem evolvesCopy pageLangChain is the easiest way to start building agents and ap

In [28]:
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")  # define chunk_size which is length of characters, and also separator.
chunks = text_splitter.split_documents(document)
print(len(chunks))

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [29]:
chunks[5].page_content   # take a look at any chunk's page content

NameError: name 'chunks' is not defined

In [30]:
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames

embed_params = {
    EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
    EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}

In [31]:
from langchain_ibm import WatsonxEmbeddings

watsonx_embedding = WatsonxEmbeddings(
    model_id="ibm/slate-125m-english-rtrvr-v2",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
    params=embed_params,
)

ValidationError: 1 validation error for WatsonxEmbeddings
  Value error, Did not find 'api_key' or 'token', please add an environment variable `WATSONX_API_KEY` or 'WATSONX_TOKEN' which contains it, or pass 'api_key' or 'token' as a named parameter. [type=value_error, input_value={'model_id': 'ibm/slate-1...: {'input_text': True}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

In [32]:
texts = [text.page_content for text in chunks]

embedding_result = watsonx_embedding.embed_documents(texts)
embedding_result[0][:5]

NameError: name 'chunks' is not defined

In [33]:
from langchain.vectorstores import Chroma

docsearch = Chroma.from_documents(chunks, watsonx_embedding)

ModuleNotFoundError: No module named 'langchain.vectorstores'

In [34]:
query = "Langchain"
docs = docsearch.similarity_search(query)
print(docs[0].page_content)

NameError: name 'docsearch' is not defined

In [35]:
retriever = docsearch.as_retriever()

NameError: name 'docsearch' is not defined

In [36]:
docs = retriever.invoke("Langchain")
docs[0]

NameError: name 'retriever' is not defined

In [37]:
from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore

# Set two splitters. One is with big chunk size (parent) and one is with small chunk size (child)
parent_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=20, separator='\n')

vectorstore = Chroma(
    collection_name="split_parents", embedding_function=watsonx_embedding
)

# The storage layer for the parent documents
store = InMemoryStore()

ModuleNotFoundError: No module named 'langchain.retrievers'

In [38]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

NameError: name 'ParentDocumentRetriever' is not defined

In [39]:
retriever.add_documents(document)

NameError: name 'retriever' is not defined

In [40]:
len(list(store.yield_keys()))

NameError: name 'store' is not defined

In [41]:
sub_docs = vectorstore.similarity_search("Langchain")

NameError: name 'vectorstore' is not defined

In [42]:
print(sub_docs[0].page_content)

NameError: name 'sub_docs' is not defined

In [43]:
retrieved_docs = retriever.invoke("Langchain")

NameError: name 'retriever' is not defined

In [44]:
print(retrieved_docs[0].page_content)

NameError: name 'retrieved_docs' is not defined

In [45]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(llm=granite_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "what is this paper discussing?"
qa.invoke(query)

ModuleNotFoundError: No module named 'langchain.chains'

In [46]:
from langchain.memory import ChatMessageHistory

chat = granite_llm

history = ChatMessageHistory()

history.add_ai_message("hi!")

history.add_user_message("what is the capital of France?")

ModuleNotFoundError: No module named 'langchain.memory'

In [47]:
history.messages

NameError: name 'history' is not defined

In [48]:
ai_response = chat.invoke(history.messages)
ai_response

NameError: name 'chat' is not defined

In [49]:
history.add_ai_message(ai_response)
history.messages

NameError: name 'history' is not defined

In [51]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

conversation = ConversationChain(
    llm=granite_llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

ModuleNotFoundError: No module named 'langchain.memory'

In [52]:
conversation.invoke(input="Hello, I am a little cat. Who are you?")

NameError: name 'conversation' is not defined

In [53]:
conversation.invoke(input="What can you do?")

NameError: name 'conversation' is not defined

In [54]:
conversation.invoke(input="Who am I?.")

NameError: name 'conversation' is not defined

In [55]:
from langchain.chains import LLMChain

template = """Your job is to come up with a classic dish from the area that the users suggests.
                {location}
                
                YOUR RESPONSE:
"""
prompt_template = PromptTemplate(template=template, input_variables=['location'])

# chain 1
location_chain = LLMChain(llm=granite_llm, prompt=prompt_template, output_key='meal')

ModuleNotFoundError: No module named 'langchain.chains'

In [56]:
location_chain.invoke(input={'location':'China'})

NameError: name 'location_chain' is not defined

In [57]:
from langchain.chains import SequentialChain

template = """Given a meal {meal}, give a short and simple recipe on how to make that dish at home.

                YOUR RESPONSE:
"""
prompt_template = PromptTemplate(template=template, input_variables=['meal'])

# chain 2
dish_chain = LLMChain(llm=granite_llm, prompt=prompt_template, output_key='recipe')

ModuleNotFoundError: No module named 'langchain.chains'

In [58]:
template = """Given the recipe {recipe}, estimate how much time I need to cook it.

                YOUR RESPONSE:
"""
prompt_template = PromptTemplate(template=template, input_variables=['recipe'])

# chain 3
recipe_chain = LLMChain(llm=granite_llm, prompt=prompt_template, output_key='time')

NameError: name 'LLMChain' is not defined

In [59]:
# overall chain
overall_chain = SequentialChain(chains=[location_chain, dish_chain, recipe_chain],
                                      input_variables=['location'],
                                      output_variables=['meal', 'recipe', 'time'],
                                      verbose= True)

NameError: name 'SequentialChain' is not defined

In [60]:
from pprint import pprint

pprint(overall_chain.invoke(input={'location':'China'}))

NameError: name 'overall_chain' is not defined

In [61]:
from langchain.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm=granite_llm, chain_type="stuff", verbose=False)
response = chain.invoke(web_data)

ModuleNotFoundError: No module named 'langchain.chains'

In [62]:
print(response['output_text'])

NameError: name 'response' is not defined

In [63]:
from langchain.agents import Tool
from langchain_experimental.utilities import PythonREPL

python_repl = PythonREPL()

ImportError: cannot import name 'Tool' from 'langchain.agents' (C:\Users\Gaten\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\langchain\agents\__init__.py)

In [64]:
python_repl.run("a = 3; b = 1; print(a+b)")

NameError: name 'python_repl' is not defined

In [65]:
from langchain_experimental.tools import PythonREPLTool

tools = [PythonREPLTool()]

In [66]:
from langchain.agents import create_react_agent
from langchain import hub
from langchain.agents import AgentExecutor

instructions = """You are an agent designed to write and execute python code to answer questions.
You have access to a python REPL, which you can use to execute python code.
If you get an error, debug your code and try again.
Only use the output of your code to answer the question. 
You might know the answer without running any code, but you should still run the code to get the answer.
If it does not seem like you can write code to answer the question, just return "I don't know" as the answer.
"""

# here you will use the prompt directly from the langchain hub
base_prompt = hub.pull("langchain-ai/react-agent-template")
prompt = base_prompt.partial(instructions=instructions)

ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (C:\Users\Gaten\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\langchain\agents\__init__.py)

In [67]:
agent = create_react_agent(granite_llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)  # tools were defined in the toolkit part above

NameError: name 'create_react_agent' is not defined

In [68]:
agent_executor.invoke(input = {"input": "What is the 3rd fibonacci number?"})

NameError: name 'agent_executor' is not defined